# 4. Streaming Graph Execution

Contrast with LangChain's token-level streaming (notebook 8 in the `langchain`
folder): a graph's `.astream(stream_mode="updates")` streams one event *per node*
as it finishes — you watch the graph's control flow execute step by step, not the
text of a single response appear character by character. Reuses the same
fact -> joke graph shape as notebook 1 so the two streaming styles can be
compared on identical underlying work.

**Prerequisites:** Ollama running locally with `llama3.2` pulled.

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

from models.chat_models.ollama_models import SupportedModel, get_chat_model


class FactJokeState(TypedDict):
    topic: str
    fact: str
    joke: str


llm = get_chat_model(SupportedModel.llama3_2)


def generate_fact(state: FactJokeState) -> dict:
    response = llm.invoke(f"Give one interesting, concise fact about {state['topic']}.")
    return {"fact": response.content}


def generate_joke(state: FactJokeState) -> dict:
    response = llm.invoke(f"Write one short joke based on this fact: {state['fact']}")
    return {"joke": response.content}


graph = StateGraph(FactJokeState)
graph.add_node("generate_fact", generate_fact)
graph.add_node("generate_joke", generate_joke)
graph.add_edge(START, "generate_fact")
graph.add_edge("generate_fact", "generate_joke")
graph.add_edge("generate_joke", END)
compiled = graph.compile()

## Stream one event per node

In [ ]:
async for update in compiled.astream({"topic": "penguins", "fact": "", "joke": ""}, stream_mode="updates"):
    for node_name, output in update.items():
        print(f"[{node_name}] {output}")

## 🧪 Playground

**1. Try `stream_mode="values"`** instead of `"updates"` — what's the difference in what gets yielded?

In [ ]:
# TODO: async for state in compiled.astream(..., stream_mode='values'): print(state)


**2. Time each node** — record a timestamp before/after each yielded update to see how long `generate_fact` vs `generate_joke` actually takes.

In [ ]:
# TODO: use time.monotonic() around the astream loop


**3. A three-node graph** — add a `generate_pun` node (like notebook 1's Playground) and watch it show up as a third streamed event.

In [ ]:
# TODO: rebuild the graph with a third node and re-stream
